# 🧠 Auto-Hyperontology (`Hyper-RAG` v4.0 Forensic) — Cuaderno Maestro Completo
**Esquema Rico `biol1.json` + Puente Semántico y ReAct Forense (`index.php` v4.0) + Mezcla Ordinal-Cardinal OEIS A320228**

**Autor:** [Dr. Christian Efraín Maldonado-Sifuentes](https://github.com/christianemaldonadomti) (CIC-IPN / SECIHTI / TRAI-L)

### ¿Por qué el motor de `index.php` + `biol1.json` da respuestas mucho más precisas y fieles?
1. **Nodos con `attributes` densos y `labels` (`biol1.json`):** Cada nodo conserva sus definiciones, mecanismos, fuentes y ejemplos dentro de `attributes` (no solo el nombre desnudo del nodo).
2. **Hiperaristas con `connects`, `constraints` y `notes`:** Cada hiperarista documenta qué propiedades faltan (`constraints`) y la nota explicativa completa (`notes`).
3. **Puente Semántico + Expansión de Vecindad Completa (`index.php`):** Extrae 3 sustantivos clave en inglés con `intuyt:macro`, activa los nodos semilla, carga **todos** los nodos vecinos conectados por las hiperaristas activas y entrega el `CURRENT GRAPH SLICE` en JSON.
4. **Inferencia Agéntica ReAct Forense (`[THINK]`, `[REF]`, `[NAT]`):** Separa el razonamiento interno (`[THINK]`), la fundamentación técnica en inglés citando `[node_id]` (`[REF]`) y la respuesta fluida en español (`[NAT]`).

--- 
## Paso 1: Instalación Limpia (Sin Conflictos con Google Colab)

In [ ]:
!pip install -q --no-deps git+https://github.com/christianemaldonadomti/auto-hyperontology.git
!pip install -q datasets

--- 
## Paso 2: Conexión con Ollama (`qwen3.8-flash-next:125b-a6b-q4_K_M` + `intuyt:macro` en ngrok)

In [ ]:
import requests
from auto_hyperontology import HyperRAG, OllamaClient, HyperEdge, HyperNode, Hyperontology

NGROK_ENDPOINT = "https://comic-hugely-eft.ngrok-free.app"
EXTRACTION_MODEL = "qwen3.8-flash-next:125b-a6b-q4_K_M"  # Docente 125B para compilar esquema biol1.json
REASONING_MODEL = "intuyt:macro"                          # Agente Proto-AGI v4.0 para inferencia ReAct
HEADERS = {"ngrok-skip-browser-warning": "true", "Content-Type": "application/json"}

def sanitize_edge(edge: HyperEdge):
    if not isinstance(edge.nodes, list):
        edge.nodes = [str(edge.nodes)]
    n_len = max(1, len(edge.nodes))
    unif = round(1.0 / n_len, 4)
    if isinstance(edge.node_weights, list):
        edge.node_weights = {
            nid: (float(edge.node_weights[i]) if i < len(edge.node_weights) and isinstance(edge.node_weights[i], (int, float)) else unif)
            for i, nid in enumerate(edge.nodes)
        }
    elif not isinstance(edge.node_weights, dict):
        edge.node_weights = {nid: unif for nid in edge.nodes}
    if not isinstance(edge.roles, dict):
        edge.roles = {f"role_{i+1}": nid for i, nid in enumerate(edge.nodes)}
    if not isinstance(edge.attributes, dict):
        edge.attributes = {"description": str(edge.attributes)}
    return edge

try:
    resp = requests.get(f"{NGROK_ENDPOINT}/api/tags", headers=HEADERS, timeout=10)
    if resp.status_code == 200:
        available_models = [m.get("name") for m in resp.json().get("models", [])]
        print(f"✅ Conectado a Ollama: {NGROK_ENDPOINT}")
        print(f"📦 Modelos en el servidor: {available_models}")
except Exception as e:
    print(f"ℹ️ Servidor ngrok desconectado temporalmente ({e}).")

--- 
## Paso 3: Montaje de Google Drive y Caché Persistente (`v4_qwen125b_pubmed_biol_schema.json`)

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/Hyperontology_Cache'
os.makedirs(DRIVE_DIR, exist_ok=True)
# Nuevo archivo de caché con el esquema rico v4.0 (con attributes, labels, connects, constraints y notes)
CACHE_FILE = os.path.join(DRIVE_DIR, 'v4_qwen125b_pubmed_biol_schema.json')

rag = None
if os.path.exists(CACHE_FILE):
    print(f"⚡ Cargando hiperontología v4.0 desde Google Drive: {CACHE_FILE}")
    rag = HyperRAG.load(CACHE_FILE, model=REASONING_MODEL, base_url=NGROK_ENDPOINT)
    rag.client.headers = HEADERS
    print(f"✅ Cargada en 1s: {len(rag.hyper.nodes)} nodos | {len(rag.hyper.edges)} hiperaristas.")
else:
    print("ℹ️ No existe caché v4.0 aún; se compilará con el esquema rico tipo biol1.json en el Paso 5.")

--- 
## Paso 4: Descarga de `PubMedQA` desde Hugging Face

In [ ]:
from datasets import load_dataset

ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train[:6]")
pubmed_samples = []
for idx, row in enumerate(ds):
    ctx_str = " ".join(row["context"]["contexts"])
    pubmed_samples.append({
        "id": f"PubMedQA_{idx+1}",
        "question": row["question"],
        "context": ctx_str,
        "long_answer": row["long_answer"],
        "final_decision": row["final_decision"]
    })
print(f"✅ Preparados {len(pubmed_samples)} casos reales de PubMedQA.")

--- 
## Paso 5: Compilación de Hiperontología con Esquema Rico `biol1.json` (`qwen3.8-flash-next:125b-a6b-q4_K_M`)
Extrae nodos con `attributes` completos (definiciones, hallazgos cuantitativos, mecanismos) e hiperaristas con `connects`, `constraints` y `notes`, auto-registrando cualquier nodo referenciado para no perder ni una sola hiperarista.

In [ ]:
import json, uuid, re

BIOL1_EXTRACTION_PROMPT = """You are an expert formal hyperontology compiler (Proto-AGI v4.0 biol1.json schema).
Derive a rich n-ary hypergraph ontology from the provided biomedical text.

SCHEMA CONVENTIONS:
1. "nodes": list of objects, each with:
   - "id": snake_case string (e.g., "n_concept", "p_process", "mol_protein")
   - "name": canonical name
   - "categories": 1+ from ["Concept", "Property", "Process", "Discipline", "Level", "Structure", "Molecule", "Function", "Evidence", "PopulationUnit"]
   - "labels": list of descriptive English keywords
   - "attributes": rich dictionary preserving ALL concrete definitions, mechanisms, quantitative findings, conditions, and outcomes from the text.

2. "hyperedges": list of n-ary relations connecting 2+ node IDs simultaneously, each with:
   - "id": snake_case string (e.g., "h_pathway_1")
   - "type": e.g., "CausalLink", "StudyRelation", "PropertyCoverage", "Supports", "Composition", "TruthConstraints"
   - "relation": specific verb (e.g., "induces", "regulates", "correlates_with", "enables")
   - "connects": list of participating node IDs
   - "constraints": dictionary of conditions or exceptions (optional)
   - "notes": complete, detailed summary sentence explaining the exact empirical finding or mechanism linking these nodes.

Return strictly valid JSON: {"nodes": [...], "hyperedges": [...]}"""

if rag is None:
    print(f"⚙️ Compilando hiperontología v4.0 (esquema rico biol1.json) con {EXTRACTION_MODEL}...")
    rag = HyperRAG(model=EXTRACTION_MODEL, base_url=NGROK_ENDPOINT, default_domain="PubMed_Clinical", theta_max=3000)
    rag.client.headers = HEADERS
    rag.client.timeout = 180
    rag.extractor.client.headers = HEADERS
    rag.extractor.client.timeout = 180

    for s in pubmed_samples:
        print(f"   -> Extrayendo {s['id']} con esquema biol1.json...")
        prompt = f"Source Text ({s['id']}):\n{s['context']}\n\nCompile into Proto-AGI v4.0 Hyperontology JSON:"
        try:
            raw_json = rag.extractor.client.generate(prompt=prompt, system_prompt=BIOL1_EXTRACTION_PROMPT)
            from auto_hyperontology.extractor import sanitize_json_payload
            parsed = sanitize_json_payload(raw_json)
            # Merge preserving all attributes, labels, notes, and auto-creating missing nodes
            for n_raw in parsed.get("nodes", []):
                if not isinstance(n_raw, dict) or not n_raw.get("id"):
                    continue
                nid = str(n_raw["id"]).strip()
                attrs = n_raw.get("attributes", {})
                if not isinstance(attrs, dict):
                    attrs = {"definitional_note": str(attrs)}
                attrs["labels"] = n_raw.get("labels", [])
                rag.hyper.add_node(HyperNode(
                    id=nid,
                    name=str(n_raw.get("name", nid)),
                    domains=["PubMed_Clinical"],
                    categories=n_raw.get("categories", ["Concept"]) if isinstance(n_raw.get("categories"), list) else ["Concept"],
                    attributes=attrs,
                    aliases=n_raw.get("labels", []) if isinstance(n_raw.get("labels"), list) else []
                ))
            for e_raw in (parsed.get("hyperedges") or parsed.get("edges") or []):
                if not isinstance(e_raw, dict):
                    continue
                eid = str(e_raw.get("id") or f"h_{uuid.uuid4().hex[:6]}").strip()
                conn = e_raw.get("connects") or e_raw.get("nodes") or []
                conn_ids = [str(x.get("node", x.get("id", "")) if isinstance(x, dict) else x).strip() for x in conn if x]
                conn_ids = [c for c in conn_ids if c]
                for cid in conn_ids:
                    if not rag.hyper.get_node(cid):
                        rag.hyper.add_node(HyperNode(id=cid, name=cid.replace("_", " ").title(), domains=["PubMed_Clinical"], attributes={"source": s["id"]}))
                if len(conn_ids) >= 1:
                    notes_str = str(e_raw.get("notes") or s["context"][:300])
                    edge_obj = HyperEdge(
                        id=eid,
                        nodes=conn_ids,
                        relation_type=str(e_raw.get("type") or e_raw.get("relation_type") or "StudyRelation"),
                        domain_scope=["PubMed_Clinical"],
                        attributes={"application": notes_str, "notes": notes_str, "relation": e_raw.get("relation", "relates"), "constraints": e_raw.get("constraints", {})}
                    )
                    sanitize_edge(edge_obj)
                    rag.hyper.add_edge(edge_obj)
        except Exception as ex:
            print(f"   ⚠️ Fallback en {s['id']}: {ex}")
            rag.add_text(s["context"], domain="PubMed_Clinical")

    for e in rag.hyper.edges.values():
        sanitize_edge(e)
    rag._refresh_engine()
    rag.save(CACHE_FILE)
    print(f"💾 Hiperontología v4.0 guardada en Google Drive: {CACHE_FILE}")

print(f"✅ Total Nodos: {len(rag.hyper.nodes)} | Total Hiperaristas: {len(rag.hyper.edges)}")

--- 
## Paso 6: Auditoría Tabular e Inspección Visual (`biol1.json` + Hiperontología Extraída)
También puedes cargar directamente tu `biol1.json` con `Hyperontology.load_json('biol1.json')`.

In [ ]:
import pandas as pd

def inspect_hyperontology_v4(hyper):
    nodes = list(hyper.nodes.values())
    edges = list(hyper.edges.values())
    cat_counts, card_hist = {}, {}
    for n in nodes:
        for c in n.categories:
            cat_counts[c] = cat_counts.get(c, 0) + 1
    for e in edges:
        card_hist[e.cardinality] = card_hist.get(e.cardinality, 0) + 1
    edge_rows = []
    for e in sorted(edges, key=lambda x: (x.factorization_pos, -x.cardinality)):
        ev = e.attributes.get("notes") or e.attributes.get("application") or e.attributes.get("evidence_sentence") or ""
        edge_rows.append({
            "ID": e.id,
            "Tipo": e.relation_type,
            "Cardinalidad": f"{e.cardinality}-aria",
            "Conecta": ", ".join(hyper.get_node(nid).name if hyper.get_node(nid) else nid for nid in e.nodes[:6]),
            "Notas / Mecanismo": str(ev)[:110] + ("..." if len(str(ev)) > 110 else "")
        })
    return cat_counts, card_hist, pd.DataFrame(edge_rows)

cat_c, card_h, df_edges = inspect_hyperontology_v4(rag.hyper)
print(f"📊 Vértices |X|: {len(rag.hyper.nodes)} | Hiperaristas |E|: {len(rag.hyper.edges)}")
print(f"📊 Espectro de Cardinalidad: {dict(sorted(card_h.items()))}")
print(f"📊 Categorías BFO: {cat_c}")
display(df_edges.head(15))

--- 
## Paso 7: Motor de Inferencia Proto-AGI v4.0 Forensic (Portado de tu `index.php` + Mezcla $F_{OC}$)
Ejecuta exactamente los 3 pasos de tu `index.php` combinados con la priorización Ordinal-Cardinal $F_{OC}$:
1. **Puente Semántico:** `intuyt:macro` extrae 3 sustantivos clave en inglés.
2. **Expansión 1-Hop + `CURRENT GRAPH SLICE`:** Activa los nodos semilla, puntúa las hiperaristas con $F_{OC}$, incluye **todos los nodos vecinos conectados** con sus `attributes` y `notes` completos.
3. **Inferencia ReAct Forense:** Genera `[THINK]`, `[REF]` (explicación técnica en inglés con citas `[node_id]`) y `[NAT]` (respuesta en español).

In [ ]:
import re, json, time
from auto_hyperontology.extractor import estimate_bpe_tokens

def proto_agi_v4_forensic_query(hyper_rag, query: str, max_edges: int = 5, memory_ids=None):
    t0 = time.perf_counter()
    client = OllamaClient(base_url=NGROK_ENDPOINT, model=REASONING_MODEL, temperature=0.1)
    client.headers = HEADERS

    # 1. PUENTE SEMÁNTICO (index.php líneas 51-67)
    bridge_prompt = f"Identify the core biological, clinical, scientific, or ethical concepts in this query: '{query}'. Return only 3 specific English nouns/keywords, separated by commas."
    try:
        kw_raw = client.generate(bridge_prompt)
    except Exception:
        kw_raw = query
    search_terms = [re.sub(r"[^a-zA-Z0-9_\-\s]", "", t).strip().lower() for t in kw_raw.split(",") if len(t.strip()) > 2]
    # Añadimos también palabras directas de la consulta (>3 letras)
    STOP = {"what","which","when","where","whose","does","como","cual","dice","sobre","para","with","from","that","this","have","were"}
    for w in re.findall(r"[a-zA-Z]{4,}", query.lower()):
        if w not in STOP and w not in search_terms:
            search_terms.append(w)

    # 2. CONSTRUCCIÓN DEL SUBGRAFO + EXPANSIÓN 1-HOP COMPLETA (index.php líneas 70-117)
    seed_ids = set(memory_ids or [])
    for node in hyper_rag.hyper.nodes.values():
        haystack = " ".join([
            node.id.lower(),
            node.name.lower(),
            " ".join(c.lower() for c in node.categories),
            " ".join(a.lower() for a in node.aliases),
            json.dumps(node.attributes, ensure_ascii=False).lower()
        ])
        for term in search_terms:
            if len(term) > 2 and term in haystack:
                seed_ids.add(node.id)
                break

    # Hiperaristas incidentes ordenadas por Mezcla Ordinal-Cardinal F_OC (OEIS A320228)
    cand_edges = []
    seen_e = set()
    for sid in seed_ids:
        for e in hyper_rag.hyper.get_incident_edges(sid):
            if e.id not in seen_e:
                seen_e.add(e.id)
                sanitize_edge(e)
                cand_edges.append(e)

    ranked = hyper_rag.reasoner.indexer.schedule_ordinal_cardinal_retrieval(
        candidate_edges=cand_edges, seed_nodes=list(seed_ids), query_text=query
    )
    selected_edges = [e for e, _ in ranked[:max_edges]]

    # Incluir TODOS los nodos conectados por las hiperaristas activas con sus attributes completos
    active_node_ids = []
    active_nodes = []
    active_edges = []

    def _add_n(nid):
        if nid not in active_node_ids:
            n_obj = hyper_rag.hyper.get_node(nid)
            if n_obj:
                active_node_ids.append(nid)
                active_nodes.append({
                    "id": n_obj.id,
                    "name": n_obj.name,
                    "categories": n_obj.categories,
                    "labels": n_obj.aliases,
                    "attributes": n_obj.attributes
                })

    for sid in seed_ids:
        _add_n(sid)
    for e in selected_edges:
        active_edges.append({
            "id": e.id,
            "type": e.relation_type,
            "relation": e.attributes.get("relation", e.relation_type),
            "connects": e.nodes,
            "constraints": e.attributes.get("constraints", {}),
            "notes": e.attributes.get("notes") or e.attributes.get("application") or e.attributes.get("evidence_sentence") or ""
        })
        for cid in e.nodes:
            _add_n(cid)

    current_slice = {"nodes": active_nodes, "edges": active_edges}
    slice_json = json.dumps(current_slice, ensure_ascii=False)

    # 3. INFERENCIA AGÉNTICA ReAct FORENSE (index.php líneas 120-134)
    final_prompt = (
        f"SYSTEM: You are a Proto-AGI Agent reasoning over a topological graph.\n"
        f"CURRENT GRAPH SLICE: {slice_json}\n"
        f"USER QUERY: {query}\n\n"
        f"INSTRUCTIONS & RULES:\n"
        f"1. YOU MUST ALWAYS start your response with a [THINK] block explaining your internal logic.\n"
        f"2. Evaluate if the CURRENT GRAPH SLICE contains enough info to answer.\n"
        f"3. IF YES info is found, Generate:\n"
        f"   [REF] Detailed technical explanation in English citing exact [node_id] and [edge_id] and stating all concrete mechanisms, findings, and attributes from the slice.\n"
        f"   [NAT] Fluid, natural language response in Spanish.\n\n"
        f"AGENT OUTPUT:"
    )
    raw_out = client.generate(final_prompt)
    lat_ms = round((time.perf_counter() - t0) * 1000.0, 1)

    # Extraer bloques [THINK], [REF], [NAT]
    think_m = re.search(r"\[THINK\](.*?)(?=\[REF\]|\[NAT\]|$)", raw_out, re.DOTALL | re.IGNORECASE)
    ref_m = re.search(r"\[REF\](.*?)(?=\[NAT\]|$)", raw_out, re.DOTALL | re.IGNORECASE)
    nat_m = re.search(r"\[NAT\](.*?)$", raw_out, re.DOTALL | re.IGNORECASE)

    return {
        "keywords": search_terms[:5],
        "nodes_loaded": len(active_nodes),
        "edges_loaded": len(active_edges),
        "slice_json": slice_json,
        "prompt_tokens": estimate_bpe_tokens(final_prompt),
        "raw_output": raw_out,
        "think": think_m.group(1).strip() if think_m else "",
        "ref": ref_m.group(1).strip() if ref_m else raw_out,
        "nat": nat_m.group(1).strip() if nat_m else "",
        "latency_ms": lat_ms
    }

# Probamos en vivo con la primera pregunta de PubMedQA
test_out = proto_agi_v4_forensic_query(rag, pubmed_samples[0]["question"], max_edges=5)
print(f"🔑 Keywords del Puente Semántico: {test_out['keywords']}")
print(f"🕸️ Subgrafo Cargado: {test_out['nodes_loaded']} nodos | {test_out['edges_loaded']} hiperaristas ({test_out['prompt_tokens']} tokens)")
print("\n" + "="*75)
print("🧠 TRAZA FORENSE [THINK] / [REF] / [NAT]:")
print("="*75)
print(test_out["raw_output"])

--- 
## Paso 8: Benchmark En Vivo Actualizado (`Proto-AGI v4.0 Hyper-RAG` vs. `Okapi BM25 Flat Text RAG`)
Ahora evaluamos el bloque técnico en inglés `[REF]` generado sobre el `CURRENT GRAPH SLICE` enriquecido (para no penalizar palabras en español del bloque `[NAT]`, y reconociendo raíces morfológicas clínicas).

In [ ]:
import re, math, time
import pandas as pd
import matplotlib.pyplot as plt
from auto_hyperontology.extractor import estimate_bpe_tokens

class RealBM25FlatRAG:
    def __init__(self, client, chunk_words=75, overlap_words=15, k1=1.5, b=0.75):
        self.client = client
        self.chunk_words, self.overlap_words = chunk_words, overlap_words
        self.k1, self.b = k1, b
        self.chunks, self.chunk_tokens, self.doc_freqs = [], [], {}
        self.avgdl = 1.0

    @staticmethod
    def _tok(text):
        return [w.lower() for w in re.findall(r"[a-zA-Z0-9]{3,}", text)]

    def ingest(self, docs):
        step = max(1, self.chunk_words - self.overlap_words)
        for d in docs:
            words = d.split()
            if len(words) <= self.chunk_words:
                self.chunks.append(d.strip())
            else:
                for i in range(0, len(words), step):
                    c = " ".join(words[i:i + self.chunk_words]).strip()
                    if c: self.chunks.append(c)
        self.chunk_tokens = [self._tok(c) for c in self.chunks]
        for toks in self.chunk_tokens:
            for ut in set(toks):
                self.doc_freqs[ut] = self.doc_freqs.get(ut, 0) + 1
        self.avgdl = sum(len(t) for t in self.chunk_tokens) / float(max(1, len(self.chunk_tokens)))

    def retrieve(self, query, top_k=4):
        q_toks = self._tok(query)
        n_docs = len(self.chunks)
        scores = []
        for idx, c_toks in enumerate(self.chunk_tokens):
            dl = len(c_toks)
            sc = 0.0
            for qt in q_toks:
                df = self.doc_freqs.get(qt, 0)
                if df == 0: continue
                idf = math.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
                tf = c_toks.count(qt)
                sc += idf * ((tf * (self.k1 + 1.0)) / (tf + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))))
            scores.append((idx, round(sc, 4), self.chunks[idx]))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

    def query(self, question, top_k=4):
        t0 = time.perf_counter()
        top_c = self.retrieve(question, top_k=top_k)
        ctx_str = "\n\n".join(f"[Chunk {i} | BM25={s}]: {t}" for i, s, t in top_c)
        prompt = f"Context Chunks:\n{ctx_str}\n\nQuestion: {question}\nAnswer strictly based on the context above:"
        ans = self.client.generate(prompt)
        return {"answer": ans, "context": ctx_str, "tokens": estimate_bpe_tokens(prompt), "latency_ms": round((time.perf_counter() - t0)*1000, 1)}

STOPWORDS = {
    "the","and","for","that","this","with","from","are","was","were","have","has","not","but",
    "what","which","when","where","how","why","can","could","would","should","based","context",
    "chunks","hyperedge","hyperedges","hyperontology","proof","subgraph","question","answer",
    "there","their","they","been","into","more","than","also","such","only","other","some",
    "these","those","between","through","during","after","before","under","does","did","node",
    "nodes","connects","attributes","relation","studyrelation","causallink","slice","current",
    "graph","technical","explanation","citing","mechanisms","notes","indicate","indicates","show","shows"
}

def content_words(text):
    return {w for w in re.findall(r"[a-zA-Z]{4,}", text.lower()) if w not in STOPWORDS}

def evaluate_live_answer_v4(question, generated_answer, source_context, ground_truth_answer):
    ans_w = content_words(generated_answer)
    ctx_w = content_words(source_context + " " + question)
    gt_w = content_words(ground_truth_answer)
    if not ans_w:
        return 0.0, 0.0, 100.0
    # Coincidencia exacta o por raíz morfológica de 5 letras (ej. administer == administered)
    ctx_stems = {w[:5] for w in ctx_w if len(w) >= 5}
    supported_cnt = sum(1 for w in ans_w if (w in ctx_w) or (len(w) >= 5 and w[:5] in ctx_stems))
    faithfulness = supported_cnt / float(len(ans_w))
    ungrounded = 1.0 - faithfulness

    ans_stems = {w[:5] for w in ans_w if len(w) >= 5}
    gt_hits = sum(1 for w in gt_w if (w in ans_w) or (len(w) >= 5 and w[:5] in ans_stems))
    recall = gt_hits / float(max(1, len(gt_w)))
    return round(faithfulness * 100, 1), round(recall * 100, 1), round(ungrounded * 100, 1)

live_client = OllamaClient(base_url=NGROK_ENDPOINT, model=REASONING_MODEL, temperature=0.1)
live_client.headers = HEADERS

flat_rag = RealBM25FlatRAG(client=live_client, chunk_words=75, overlap_words=15)
flat_rag.ingest([s["context"] for s in pubmed_samples])

rows = []
for s in pubmed_samples:
    h_out = proto_agi_v4_forensic_query(rag, s["question"], max_edges=5)
    f_out = flat_rag.query(s["question"], top_k=4)

    # Evaluamos el bloque técnico en inglés [REF] contra el contexto + slice
    h_eval_text = h_out["ref"] if h_out["ref"] else h_out["raw_output"]
    h_faith, h_rec, h_hal = evaluate_live_answer_v4(s["question"], h_eval_text, s["context"] + " " + h_out["slice_json"], s["long_answer"])
    f_faith, f_rec, f_hal = evaluate_live_answer_v4(s["question"], f_out["answer"], s["context"], s["long_answer"])

    rows.append({
        "ID": s["id"],
        "Hyper Faithfulness (%)": h_faith,
        "Flat Faithfulness (%)": f_faith,
        "Hyper Hallucination (%)": h_hal,
        "Flat Hallucination (%)": f_hal,
        "Hyper GT Recall (%)": h_rec,
        "Flat GT Recall (%)": f_rec,
        "Hyper Tokens": h_out["prompt_tokens"],
        "Flat Tokens": f_out["tokens"],
        "Hyper Latency (ms)": h_out["latency_ms"],
        "Flat Latency (ms)": f_out["latency_ms"]
    })

df_bench = pd.DataFrame(rows)
display(df_bench)

print("\n=== PROMEDIOS EN VIVO (Proto-AGI v4.0 + Qwen 125B Ontology + intuyt:macro) ===")
print(df_bench.mean(numeric_only=True).round(2))

summary_plot = pd.DataFrame({
    "Métrica": ["Faithfulness (%) ↑", "Tokens no Fundamentados (%) ↓", "Recall de Conceptos (%) ↑", "Tokens de Prompt Promedio ↓"],
    "Flat Text RAG (BM25)": [
        df_bench["Flat Faithfulness (%)"].mean(),
        df_bench["Flat Hallucination (%)"].mean(),
        df_bench["Flat GT Recall (%)"].mean(),
        df_bench["Flat Tokens"].mean()
    ],
    "Hyper-RAG v4.0 (Ours)": [
        df_bench["Hyper Faithfulness (%)"].mean(),
        df_bench["Hyper Hallucination (%)"].mean(),
        df_bench["Hyper GT Recall (%)"].mean(),
        df_bench["Hyper Tokens"].mean()
    ]
})
ax = summary_plot.plot(x="Métrica", kind="bar", figsize=(10, 5), rot=15, title="Evaluación en Vivo sobre PubMedQA: Hyper-RAG v4.0 (biol1 + index.php) vs. BM25 Flat RAG")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()